1. Phần câu hỏi trắc nghiệm

Q1. Trung vị số ngày giữa 2 lần mua liên tiếp của khách hàng có nhiều hơn 1 đơn hàn

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv('../data/processed/orders_clean.csv', parse_dates=['order_date'])
multi_order_customer = orders.groupby('customer_id').filter(lambda x: len(x) > 1)

multi_order_customer = multi_order_customer.sort_values(['customer_id', 'order_date'])

#tinh khoang cach
multi_order_customer['gap'] = multi_order_customer.groupby('customer_id')['order_date'].diff().dt.days

print("Q1:", multi_order_customer['gap'].median())

Q1: 144.0


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [3]:
products = pd.read_csv('../data/processed/products_clean.csv')

# ti suat loi nhuan
products['margin'] = (products['price'] - products['cogs']) / products['price']

# gom nhom, tinh mean va tim max
print("Q2:", products.groupby('segment')['margin'].mean().idxmax())

Q2: Standard


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [4]:
returns = pd.read_csv('../data/raw/returns.csv')

#lọc bảng
streetwear = products[products['category'] == 'Streetwear']

# join
merged_q3 = returns.merge(streetwear, on='product_id', how='inner')

# dem ly do
print("Q3:", merged_q3['return_reason'].value_counts().idxmax())

Q3: wrong_size


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [5]:
web_traffic = pd.read_csv('../data/raw/web_traffic.csv')
print("Q4:", web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin())

Q4: email_campaign


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [6]:
order_items = pd.read_csv('../data/raw/order_items.csv')

promo_ratio = order_items['promo_id'].notnull().mean() * 100
print(f"Q5: {promo_ratio:.2f}%")

Q5: 38.66%


C:\Users\OS\AppData\Local\Temp\ipykernel_12496\409386835.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('../data/raw/order_items.csv')


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [7]:
customers = pd.read_csv('../data/raw/customers.csv')

#noi 2 bang
cust_orders = customers.dropna(subset=['age_group']).merge(orders, on='customer_id', how='inner')

#tinh tong
avg_orders = cust_orders.groupby('age_group').agg(total_orders=('order_id', 'count'),unique_customers=('customer_id', 'nunique'))

avg_orders['orders_per_cust'] = avg_orders['total_orders'] / avg_orders['unique_customers']
print("Q6:", avg_orders['orders_per_cust'].idxmax())

Q6: 55+


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [12]:
geography = pd.read_csv('../data/raw/geography.csv')
payments = pd.read_csv('../data/raw/payments.csv')

#join 3 bảng với nhau
order_geo = orders.merge(geography, on='zip', how='inner')
full_q7_data = order_geo.merge(payments, on='order_id', how='inner')

#tính tổng doanh thu
print("Q7:", full_q7_data.groupby('region')['payment_value'].sum().idxmax())

Q7: East


Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [13]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
print("Q8:", cancelled_orders['payment_method'].value_counts().idxmax())

Q8: credit_card


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [14]:
#loc bang products
target_sizes = products[products['size'].isin(['S', 'M', 'L', 'XL'])]

#count so luong ban ra
sold_items = order_items.merge(target_sizes, on='product_id')
sold_by_size = sold_items['size'].value_counts()

#count so luong tra lai
returned_items = returns.merge(target_sizes, on='product_id')
returned_by_size = returned_items['size'].value_counts()

# tinh ti le
return_rate = returned_by_size / sold_by_size
print("Q9:", return_rate.idxmax())

Q9: S


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [15]:
print("Q10:", payments.groupby('installments')['payment_value'].mean().idxmax())

Q10: 6
